In [1]:
import ast
import wandb.apis.public
import itertools
import concurrent.futures
import tqdm
from pathlib import Path
import json
import os
import sys
sys.path.append("..")

import wandb
os.environ["WANDB_SILENT"] = "true" # Do not print download output.
api = wandb.Api()

In [2]:
project_id = "qnrl"

In [3]:
# List of sweep names and their IDs for reference.

# # OLD sweeps that should be deleted.
# sweep_name_id = { # name: id
#     'qnrl/FrozenLake-v1': 'oi7rm7sh',
#     'qnrl/CliffWalking-v1': '2f7ji0uu',
#     'qnrl/Acrobot-v1': '28tqdi2r',
#     'qnrl/CartPole-v1': '0xhzu1tq',
#     'c51/FrozenLake-v1': '6ebrrbwp',
#     'c51/CliffWalking-v1': '0nxxo52o',
#     'c51/Acrobot-v1': '2ti1laob',
#     'c51/CartPole-v1': 'mjqmhnd0',
# }

# NEW.
sweep_name_id = { # name: id
    # # QnRL.
    # 'qnrl/FrozenLake-v1': 'sgi6zb0t',
    # 'qnrl/CliffWalking-v1': 'h4g5er22',
    # 'qnrl/Acrobot-v1': 'sd4va92r',
    # 'qnrl/CartPole-v1': '4ykq4hvw',
    
    # C51.
    'c51/FrozenLake-v1': '3exa9jvo',
    'c51/CliffWalking-v1': '1nlfftsy',
    'c51/Acrobot-v1': 's3xwn85k',
    'c51/CartPole-v1': '7raha9om',
}

In [4]:
def get_run_config_dict(run: wandb.apis.public.Run):
    """Compute run config hash from original run program command string."""
    # Download the run metadata file, load as JSON dictionary, and then delete the local downloaded file.
    metadata_file = run.file("wandb-metadata.json").download(replace=True)
    metadata = json.load(metadata_file)
    if os.path.exists(metadata_file.name):
        os.remove(metadata_file.name)
    
    # Get config string from run program command arguments.
    # Format is: --config="{...}"
    cfg_str = metadata['args'][1].split("--config=")[1]
    # Convert this string to dictionary via literal evaluation.
    cfg_dict = ast.literal_eval(cfg_str)
    return cfg_dict

In [5]:
root_sweeps = Path("../wandb-sweep-data/")
root_sweeps.mkdir(parents=True, exist_ok=True)

In [6]:
def download_run_data(root: Path, run: wandb.apis.public.Run):
    """Download files for the given run. Returns a list of functions that when called will download the associated file."""
    root_run = root / f"run-{str(run.id)}"
    root_run.mkdir(parents=True, exist_ok=True)
    return [
        # Download metadata.
        lambda: run.file("wandb-metadata.json").download(root=str(root_run), replace=True),
        # Download metrics history.
        lambda: api.artifact(name=f"{project_id}/run-{run.id}-history:latest", type="wandb-history").download(root=str(root_run / "history")),
        # Download saved models.
        lambda: api.artifact(name=f"{project_id}/run-{run.id}-models:latest", type="model").download(root=str(root_run / "models")),
    ]

def download_all_run_data_for_sweep(root: Path, sweep: wandb.apis.public.Sweep):
    """Download files for all runs associated with the given sweep."""
    root_sweep = root / f"sweep-{str(sweep.id)}"
    root_sweep.mkdir(parents=True, exist_ok=True)
    return list(itertools.chain.from_iterable(
        download_run_data(
            root=root_sweep,
            run=run,
        )
        for run in sweep.runs
    ))

In [ ]:
# Download all sweeps.
sweeps_to_download = list(sweep_name_id.values())

# Collect list of files and folders to download.
files_to_download = list(tqdm.tqdm(
    itertools.chain.from_iterable(
        download_all_run_data_for_sweep(
            root=root_sweeps, 
            sweep=api.sweep(f"{project_id}/{sweep_id}"),
        )
        for sweep_id in sweeps_to_download
    ),
    unit="files",
    desc="Preparing downloads",
))

# Download all files and folders for given sweeps.
num_threads = 4 # Maximum number of concurrent threads
with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
    list(tqdm.tqdm(executor.map(lambda f: f(), files_to_download), total=len(files_to_download), unit="files", desc="Downloading"))

Preparing downloads: 192files [00:08, 21.73files/s]
Downloading: 100%|██████████| 192/192 [01:45<00:00,  1.82files/s]
